# Qwen Safety Observer-Boundary Follow-up

**Question.** Does the activation observer remain preferable when the controller can use the full prompt, Qwen's own decision margin, or tail-loss selection?

**Status.** These analyses were specified after the original locked-test outcomes were known. They are secondary checks, not independent confirmation.

Experiments designed/concieved by Vijay Erramilli. Code written by Vijay Erramilli and Codex


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd

REPO = Path.cwd().resolve()
if not (REPO / 'pyproject.toml').is_file():
    REPO = REPO.parent
CONFIG = REPO / 'configs/revision/qwen_safety/qwen2_5_7b_instruct_paired_scope_v0.json'
ARTIFACTS = REPO / 'results/revision/qwen_safety/paired_scope_v0'
OUTPUT = ARTIFACTS / 'followup_v0'
assert CONFIG.is_file() and ARTIFACTS.is_dir()
{'repo': str(REPO), 'output': str(OUTPUT)}


## Plan

1. Select the existing observer families on calibration CVaR instead of mean loss.
2. Fit a fixed full-context hashing baseline.
3. Score the locked bank with Qwen's directly extracted block-minus-allow margin.
4. Bootstrap realized loss by matched pair within each test stratum.
5. Separate within-stratum discrimination from cross-stratum budget allocation.


In [ ]:
command = [
    sys.executable,
    str(REPO / 'scripts/analyze_qwen_safety_followup.py'),
    '--config', str(CONFIG),
    '--artifacts-root', str(ARTIFACTS),
    '--output-dir', str(OUTPUT),
    '--text-dimension', '2048',
    '--bootstrap-replicates', '5000',
]
subprocess.run(command, cwd=REPO, check=True, env={**os.environ, 'PYTHONPATH': str(REPO / 'src')})


## Main observer comparison


In [ ]:
metrics = pd.read_csv(OUTPUT / 'qwen_safety_followup_results.csv').set_index('observer')
reported = [
    'action-only-direct-risk',
    'activation-transformed-label-risk',
    'full-context-text-label-times-severity',
    'model-logit-margin-times-severity',
    'model-decision-times-severity',
    'exact-authorization-risk-oracle',
]
metrics.loc[reported, ['protocol_loss_mean', 'protocol_loss_cvar', 'risk_auroc', 'severity_weighted_miss_rate', 'clean_utility_retained']].round(3)


## Paired uncertainty and shift diagnosis


In [ ]:
contrasts = pd.read_csv(OUTPUT / 'qwen_safety_followup_contrasts.csv')
contrasts[['candidate', 'reference', 'mean_loss_difference', 'mean_loss_difference_ci_low', 'mean_loss_difference_ci_high', 'cvar_difference', 'cvar_difference_ci_low', 'cvar_difference_ci_high']].round(3)


In [ ]:
allocations = pd.read_csv(OUTPUT / 'qwen_safety_followup_stratum_allocations.csv')
allocations.loc[
    allocations['observer'].isin(['activation-transformed-label-risk', 'model-logit-margin-times-severity']),
    ['observer', 'family_id', 'unsafe_intervention_rate', 'share_of_global_interventions', 'mean_realized_loss'],
].round(3)


## Findings

- CVaR selection chooses the same activation layer and ridge as mean-loss selection.
- The Qwen decision margin times known severity improves both mean and tail loss over the activation observer and nearly matches the exact policy oracle.
- The full-context text observer is competitive with the activation observer, so this explicit-decision fixture cannot establish a uniquely latent safety state.
- The activation label component has AUROC 1.000 within every stratum. Its doubly-held-out loss is a cross-stratum calibration and budget-allocation failure.

**Decision.** Keep this study as an observer-interface and safety-control result. Build the next fixture in execution mode, without asking the model to emit an authorization verdict.
